# Lab 2: Building a GenAI Operational Dashboard on CloudWatch

In Lab 1, you built a multi-agent claims processing system with four specialized agents orchestrated through a Strands GraphBuilder. Your agents now analyze documents, retrieve policies, inspect for fraud, and generate claim summaries.

But how do you know if your agents are performing well in production? How much does each claim cost to process? Which agent is the bottleneck?

In this lab, you'll instrument your claims workflow with operational metrics and build a CloudWatch dashboard to answer these questions.

**What you'll build:**
- Extract real metrics from your Lab 1 graph execution results
- Publish custom GenAI metrics to CloudWatch
- Create a dashboard step-by-step with token, latency, cache, and cost widgets

**Prerequisites:** Complete Lab 1 (multi-agent system must be working)

**Time:** 30 minutes

## Why These Metrics Matter

GenAI applications have unique operational characteristics:

| Metric Category | Why It Matters |
|-----------------|----------------|
| **Token Usage** | Directly determines cost - input/output tokens have different pricing |
| **Latency** | User experience - TTFT affects perceived responsiveness |
| **Cache Efficiency** | Cost optimization - cache hits reduce costs by up to 90% |
| **Agent Cycles** | Operational health - high cycle counts indicate inefficient prompts |
| **Errors/Throttles** | Reliability - capacity planning and error handling |

## Step 1: Setup and Configuration

You'll reuse the `ClaimsAssistant` class from Lab 1 (imported from `claims_agents.py`). When you call `assistant.analyze_claim()`, the Strands SDK automatically captures per-agent metrics that Bedrock returns with each invocation.

Below: CloudWatch and Bedrock clients, the metrics namespace (`ClaimsProcessing/GenAI`), and pricing constants for Claude Haiku 4.5 — used later to calculate real-time cost estimates from token counts.


In [1]:
# Install workshop dependencies. These pins are the versions the workshop was tested on.
!pip install --quiet \
    'strands-agents[otel]==1.37.0' \
    'bedrock-agentcore-starter-toolkit>=0.1,<1.0' \
    'aws-opentelemetry-distro>=0.10,<1.0' \
    'pydantic>=2.10,<3.0' \
    'boto3>=1.35,<2.0' \
    'awscli>=1.35,<2.0' \
    'opentelemetry-api>=1.30,<2.0'



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


### One-time account setup: enable CloudWatch Transaction Search

AgentCore evaluations read agent traces from the `aws/spans` CloudWatch log group. That log group only exists after Transaction Search has been enabled on the account — a one-time setup. The cell below does it programmatically so you don't need the console.

If Transaction Search is already enabled (from a previous run or another workshop), the cell is idempotent — it will report the existing state and move on.

In [ ]:
import json as _json
import boto3

_sts = boto3.client('sts')
_account = _sts.get_caller_identity()['Account']
_region = 'us-east-1'
_logs = boto3.client('logs', region_name=_region)
_xray = boto3.client('xray', region_name=_region)

# 1. Allow X-Ray to write spans to the aws/spans and application-signals log groups
_policy_doc = _json.dumps({
    'Version': '2012-10-17',
    'Statement': [{
        'Sid': 'TransactionSearchXRayAccess',
        'Effect': 'Allow',
        'Principal': {'Service': 'xray.amazonaws.com'},
        'Action': 'logs:PutLogEvents',
        'Resource': [
            f'arn:aws:logs:{_region}:{_account}:log-group:aws/spans:*',
            f'arn:aws:logs:{_region}:{_account}:log-group:/aws/application-signals/data:*',
        ],
        'Condition': {
            'ArnLike': {'aws:SourceArn': f'arn:aws:xray:{_region}:{_account}:*'},
            'StringEquals': {'aws:SourceAccount': _account},
        },
    }],
})
_logs.put_resource_policy(policyName='TransactionSearchAccess', policyDocument=_policy_doc)
print('✓ X-Ray log delivery policy configured')

# 2. Enable Application Signals (prerequisite for Transaction Search)
_appsig = boto3.client('application-signals', region_name=_region)
try:
    _appsig.start_discovery()
    print('✓ Application Signals discovery enabled')
except _appsig.exceptions.ClientError:
    print('✓ Application Signals already enabled')

# 3. Redirect X-Ray trace segments to CloudWatch Logs (enables Transaction Search)
_current = _xray.get_trace_segment_destination().get('Destination')
if _current == 'CloudWatchLogs':
    print('✓ Transaction Search already enabled')
else:
    _xray.update_trace_segment_destination(Destination='CloudWatchLogs')
    print('✓ Transaction Search enabled — allow ~10 minutes for aws/spans to populate')


In [ ]:
import json
import boto3
from datetime import datetime, timezone
from typing import Dict, List, Any

from claims_agents import ClaimsAssistant, create_test_claims

# Initialize clients
cloudwatch = boto3.client('cloudwatch')
NAMESPACE = 'ClaimsProcessing/GenAI'
REGION = boto3.session.Session().region_name or 'us-east-1'

# Claude Haiku 4.5 pricing (per 1K tokens)
PRICING = {
    'input': 0.001,
    'output': 0.005,
    'cache_read': 0.0001,    # 90% discount on cache reads
    'cache_write': 0.00125   # Slightly higher for cache writes
}

print(f"CloudWatch namespace: {NAMESPACE}")
print(f"Region: {REGION}")

## Step 2: Process Claims and Extract Metrics

Remember from Lab 1 that your graph executes four agents: `document_analysis` → `policy_retrieval` / `inspection` → `claim_summary`. Each agent node returns a result containing metrics.

The Strands SDK captures these metrics automatically during execution:
- `accumulated_usage` - Token counts (input, output, cache read/write)
- `accumulated_metrics` - Latency in milliseconds
- `cycle_count` - Number of reasoning loops the agent took
- `tool_metrics` - Performance data for any tools the agent called

In [ ]:
def extract_metrics_from_result(result, agent_name: str) -> Dict[str, Any]:
    """Extract metrics from a Strands graph node result.
    
    Each node in your Lab 1 graph (document_analysis, policy_retrieval, etc.)
    returns a NodeResult containing metrics from the agent execution.
    """
    if not result or not hasattr(result, 'result'):
        return None
    
    node_result = result.result
    if not node_result or not hasattr(node_result, 'metrics'):
        return None
    
    metrics = node_result.metrics
    usage = metrics.accumulated_usage if hasattr(metrics, 'accumulated_usage') else {}
    perf = metrics.accumulated_metrics if hasattr(metrics, 'accumulated_metrics') else {}
    
    return {
        'agent': agent_name,
        'input_tokens': usage.get('inputTokens', 0),
        'output_tokens': usage.get('outputTokens', 0),
        'total_tokens': usage.get('totalTokens', 0),
        'cache_read_tokens': usage.get('cacheReadInputTokens', 0),
        'cache_write_tokens': usage.get('cacheWriteInputTokens', 0),
        'latency_ms': perf.get('latencyMs', 0),
        'cycle_count': metrics.cycle_count if hasattr(metrics, 'cycle_count') else 1,
        'timestamp': datetime.now(timezone.utc)
    }

In [ ]:
# Process test claims using the same workflow from Lab 1
# This time we're collecting metrics, not just processing claims
assistant = ClaimsAssistant()
test_claims = create_test_claims()
all_metrics = []

print("Processing claims through your Lab 1 graph and extracting metrics...\n")

for claim in test_claims:
    # Same analyze_claim() call as Lab 1, but now we extract per-node metrics
    result = assistant.analyze_claim(claim)

    # Extract metrics from each agent node in the graph
    for node_id, node_result in result.results.items():
        m = extract_metrics_from_result(node_result, node_id)
        if m:
            m['claim_id'] = claim.claim_id
            all_metrics.append(m)

    print(f"  Processed: {claim.claim_id}")

print(f"\nCollected {len(all_metrics)} metric records from graph execution")


## Step 3: Calculate Cost Metrics

Each agent in your Lab 1 workflow consumes tokens differently. The Document Analysis agent processes claim descriptions and evidence lists. The Policy Retrieval agent queries coverage details. Understanding per-agent costs helps you optimize the expensive ones.

Claude Haiku 4.5 pricing (per 1K tokens):

| Token type | Price | Notes |
|---|---|---|
| Input | $0.001 | Base rate |
| Output | $0.005 | 5× input — verbose responses drive cost |
| Cache write (5-min TTL) | $0.00125 | 1.25× input |
| Cache read (hit) | $0.0001 | 10× cheaper than input (~90% savings) |

**Prices shown are for global cross-region inference (the `us.*` inference profile used in Lab 1).Check the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) under Anthropic for the up-to-date values in your region.

Output tokens are the biggest cost lever — agents like `claim_summary` that produce long narratives cost roughly 5× what agents that emit short structured answers do. Prompt caching is the main way to push costs down when the same context (system prompts, tool definitions, policy documents) is reused across invocations.

In [ ]:
def calculate_cost(metrics: Dict) -> Dict[str, float]:
    """Calculate cost breakdown from token metrics."""
    # Standard token costs
    input_cost = (metrics['input_tokens'] / 1000) * PRICING['input']
    output_cost = (metrics['output_tokens'] / 1000) * PRICING['output']
    
    # Cache costs (if applicable)
    cache_read_cost = (metrics['cache_read_tokens'] / 1000) * PRICING['cache_read']
    cache_write_cost = (metrics['cache_write_tokens'] / 1000) * PRICING['cache_write']
    
    # Calculate savings from cache
    cache_savings = (metrics['cache_read_tokens'] / 1000) * (PRICING['input'] - PRICING['cache_read'])
    
    total_cost = input_cost + output_cost + cache_read_cost + cache_write_cost
    
    return {
        'total_cost': total_cost,
        'input_cost': input_cost,
        'output_cost': output_cost,
        'cache_savings': cache_savings
    }

# Add cost data to metrics
for m in all_metrics:
    costs = calculate_cost(m)
    m.update(costs)

# Display sample
if all_metrics:
    sample = all_metrics[0]
    print("Sample metrics record:")
    print(f"  Agent: {sample['agent']}")
    print(f"  Tokens: {sample['input_tokens']} in / {sample['output_tokens']} out")
    print(f"  Cache: {sample['cache_read_tokens']} read / {sample['cache_write_tokens']} write")
    print(f"  Latency: {sample['latency_ms']}ms")
    print(f"  Cost: ${sample['total_cost']:.6f}")

## Step 4: Publish Metrics to CloudWatch

CloudWatch custom metrics let you track application-specific data alongside AWS service metrics. You'll publish metrics with an `Agent` dimension so you can filter by `document_analysis`, `policy_retrieval`, `inspection`, or `claim_summary` in your dashboard.

The `put_metric_data` API accepts up to 1000 metrics per call. For production, you'd batch these calls.

In [ ]:
def publish_metrics(metrics: Dict, namespace: str = NAMESPACE):
    """Publish a single metrics record to CloudWatch."""
    timestamp = metrics.get('timestamp', datetime.now(timezone.utc))
    dimensions = [{'Name': 'Agent', 'Value': metrics['agent']}]
    
    metric_data = [
        # Token metrics
        {'MetricName': 'InputTokens', 'Value': metrics['input_tokens'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
        {'MetricName': 'OutputTokens', 'Value': metrics['output_tokens'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
        {'MetricName': 'TotalTokens', 'Value': metrics['total_tokens'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
        
        # Cache metrics
        {'MetricName': 'CacheReadTokens', 'Value': metrics['cache_read_tokens'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
        {'MetricName': 'CacheWriteTokens', 'Value': metrics['cache_write_tokens'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
        
        # Latency metrics
        {'MetricName': 'LatencyMs', 'Value': metrics['latency_ms'], 'Unit': 'Milliseconds', 'Dimensions': dimensions, 'Timestamp': timestamp},
        
        # Cost metrics
        {'MetricName': 'EstimatedCost', 'Value': metrics['total_cost'], 'Unit': 'None', 'Dimensions': dimensions, 'Timestamp': timestamp},
        {'MetricName': 'CacheSavings', 'Value': metrics['cache_savings'], 'Unit': 'None', 'Dimensions': dimensions, 'Timestamp': timestamp},
        
        # Agent loop metrics
        {'MetricName': 'CycleCount', 'Value': metrics['cycle_count'], 'Unit': 'Count', 'Dimensions': dimensions, 'Timestamp': timestamp},
    ]
    
    cloudwatch.put_metric_data(Namespace=namespace, MetricData=metric_data)

# Publish all collected metrics
print("Publishing metrics to CloudWatch...")
for m in all_metrics:
    try:
        publish_metrics(m)
    except Exception as e:
        print(f"  Error: {e}")

print(f"Published {len(all_metrics)} metric records to {NAMESPACE}")

## Step 5: Build the CloudWatch Dashboard

Now you'll create a dashboard to visualize your Lab 1 workflow performance. Each widget answers a specific operational question:

| Widget | Question It Answers |
|--------|--------------------|
| Token Usage | Which agent consumes the most tokens? |
| Latency | Which agent is the bottleneck in my graph? |
| Cache Efficiency | Am I benefiting from prompt caching? |
| Cost Tracking | What's my spend per claim? |
| Bedrock Health | Is the underlying service healthy? |
| Cycle Count | Are my prompts efficient or causing loops? |

In [ ]:
# These are the four agents from your Lab 1 graph
# document_analysis -> policy_retrieval/inspection -> claim_summary
AGENTS = ['document_analysis', 'policy_retrieval', 'inspection', 'claim_summary']

def create_token_widget(y_pos: int) -> dict:
    """Widget 1: Token usage by agent.
    
    Helps answer: Which of my Lab 1 agents consumes the most tokens?
    Expect document_analysis to have high input (processing descriptions)
    and claim_summary to have high output (generating summaries).
    """
    return {
        'type': 'metric',
        'x': 0, 'y': y_pos, 'width': 12, 'height': 6,
        'properties': {
            'title': 'Token Usage by Agent',
            'region': REGION,
            'metrics': [
                [NAMESPACE, 'InputTokens', 'Agent', agent, {'label': f'{agent} (input)'}]
                for agent in AGENTS
            ] + [
                [NAMESPACE, 'OutputTokens', 'Agent', agent, {'label': f'{agent} (output)'}]
                for agent in AGENTS
            ],
            'stat': 'Sum',
            'period': 300,
            'view': 'timeSeries',
            'stacked': False
        }
    }

def create_latency_widget(y_pos: int) -> dict:
    """Widget 2: Latency by agent.
    
    Helps answer: Which agent is the bottleneck in my graph?
    In Lab 1, policy_retrieval and inspection run in parallel,
    so the slower one determines when claim_summary can start.
    """
    return {
        'type': 'metric',
        'x': 12, 'y': y_pos, 'width': 12, 'height': 6,
        'properties': {
            'title': 'Agent Latency (ms)',
            'region': REGION,
            'metrics': [
                [NAMESPACE, 'LatencyMs', 'Agent', agent]
                for agent in AGENTS
            ],
            'stat': 'Average',
            'period': 300,
            'view': 'timeSeries'
        }
    }

print("Token and latency widgets defined")

In [ ]:
def create_cache_widget(y_pos: int) -> dict:
    """Widget 3: Cache efficiency for prompt caching.
    
    Your Lab 1 agents have system prompts that repeat on every call.
    Caching these prompts can reduce costs by 90%.
    
    Claude Haiku 4.5 caching requirements:
    - Minimum 2,048 tokens per cache checkpoint
    - 5-minute TTL (resets on each hit)
    - Cache reads cost 90% less than standard input
    """
    return {
        'type': 'metric',
        'x': 0, 'y': y_pos, 'width': 12, 'height': 6,
        'properties': {
            'title': 'Cache Efficiency (Prompt Caching)',
            'region': REGION,
            'metrics': [
                [NAMESPACE, 'CacheReadTokens', 'Agent', agent, {'label': f'{agent} cache reads'}]
                for agent in AGENTS
            ] + [
                [NAMESPACE, 'CacheWriteTokens', 'Agent', agent, {'label': f'{agent} cache writes'}]
                for agent in AGENTS
            ],
            'stat': 'Sum',
            'period': 300,
            'view': 'timeSeries',
            'annotations': {
                'horizontal': [{
                    'label': 'Min cache checkpoint (2048 tokens)',
                    'value': 2048
                }]
            }
        }
    }

def create_cost_widget(y_pos: int) -> dict:
    """Widget 4: Cost tracking per agent.
    
    Shows estimated cost and cache savings for each Lab 1 agent.
    Use this to identify which agents to optimize first.
    """
    return {
        'type': 'metric',
        'x': 12, 'y': y_pos, 'width': 12, 'height': 6,
        'properties': {
            'title': 'Estimated Cost & Cache Savings ($)',
            'region': REGION,
            'metrics': [
                [NAMESPACE, 'EstimatedCost', 'Agent', agent, {'label': f'{agent} cost'}]
                for agent in AGENTS
            ] + [
                [NAMESPACE, 'CacheSavings', 'Agent', agent, {'label': f'{agent} savings', 'color': '#2ca02c'}]
                for agent in AGENTS
            ],
            'stat': 'Sum',
            'period': 300,
            'view': 'timeSeries'
        }
    }

print("Cache and cost widgets defined")

In [ ]:
def create_bedrock_native_widget(y_pos: int) -> dict:
    """Widget 5: Bedrock service health metrics.
    
    These metrics come from the AWS/Bedrock namespace (auto-published by AWS).
    They show the health of the underlying Bedrock service your Lab 1 agents use.
    """
    model_id = 'anthropic.claude-haiku-4-5-20251001-v1:0'
    return {
        'type': 'metric',
        'x': 0, 'y': y_pos, 'width': 24, 'height': 6,
        'properties': {
            'title': 'Bedrock Service Health (AWS/Bedrock)',
            'region': REGION,
            'metrics': [
                ['AWS/Bedrock', 'Invocations', 'ModelId', model_id, {'label': 'Invocations', 'stat': 'Sum'}],
                ['AWS/Bedrock', 'InvocationLatency', 'ModelId', model_id, {'label': 'Latency (ms)', 'stat': 'Average', 'yAxis': 'right'}],
                ['AWS/Bedrock', 'InvocationClientErrors', 'ModelId', model_id, {'label': 'Client Errors', 'stat': 'Sum', 'color': '#ff7f0e'}],
                ['AWS/Bedrock', 'InvocationServerErrors', 'ModelId', model_id, {'label': 'Server Errors', 'stat': 'Sum', 'color': '#d62728'}],
                ['AWS/Bedrock', 'InvocationThrottles', 'ModelId', model_id, {'label': 'Throttles', 'stat': 'Sum', 'color': '#9467bd'}]
            ],
            'period': 300,
            'view': 'timeSeries',
            'yAxis': {'left': {'min': 0}, 'right': {'min': 0}}
        }
    }

def create_cycle_widget(y_pos: int) -> dict:
    """Widget 6: Agent reasoning cycles.
    
    Each agent in Lab 1 may take multiple reasoning cycles to complete.
    High cycle counts (>3) suggest the agent prompt needs refinement.
    """
    return {
        'type': 'metric',
        'x': 0, 'y': y_pos, 'width': 12, 'height': 6,
        'properties': {
            'title': 'Agent Reasoning Cycles',
            'region': REGION,
            'metrics': [
                [NAMESPACE, 'CycleCount', 'Agent', agent]
                for agent in AGENTS
            ],
            'stat': 'Average',
            'period': 300,
            'view': 'timeSeries'
        }
    }

print("Bedrock native and cycle widgets defined")

## Step 6: Create the Dashboard

Assemble all widgets and deploy to CloudWatch. The dashboard will show metrics for all four agents from your Lab 1 graph: document analysis, policy retrieval, inspection, and claim summary.

In [ ]:
DASHBOARD_NAME = 'GenAI-Claims-Processing'

# Assemble dashboard with all widgets
dashboard_body = {
    'widgets': [
        create_token_widget(0),
        create_latency_widget(0),
        create_cache_widget(6),
        create_cost_widget(6),
        create_bedrock_native_widget(12),
        create_cycle_widget(18)
    ]
}

# Create the dashboard
try:
    cloudwatch.put_dashboard(
        DashboardName=DASHBOARD_NAME,
        DashboardBody=json.dumps(dashboard_body)
    )
    print(f"Dashboard '{DASHBOARD_NAME}' created successfully!")
    print(f"\nView your dashboard:")
    print(f"https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name={DASHBOARD_NAME}")
except Exception as e:
    print(f"Error creating dashboard: {e}")

## Step 7: View Your Dashboards

You now have two CloudWatch dashboards to monitor your claims workflow:

### 1. Your custom dashboard: `GenAI-Claims-Processing`

Everything you built in this lab — per-agent token usage, latency, cache efficiency, cost, Bedrock health, and cycle counts.

Open it here (the same URL is printed by Step 6):
```
https://console.aws.amazon.com/cloudwatch/home?region=<REGION>#dashboards:name=GenAI-Claims-Processing
```
Or in the console: **CloudWatch → Dashboards → GenAI-Claims-Processing**.

### 2. The built-in GenAI Observability dashboard

AWS automatically provides a dashboard that aggregates traces, sessions, and token usage for any Bedrock / AgentCore workload in your account — no setup needed.

Open it here:
```
https://console.aws.amazon.com/cloudwatch/home?region=<REGION>#gen-ai-observability/agent-core/agents
```
Or in the console: **CloudWatch → GenAI Observability → Bedrock AgentCore**.

This view becomes much richer in Lab 3 after you deploy the agent to AgentCore Runtime and enable CloudWatch Transaction Search — sessions, traces, and agent-level aggregates will appear automatically.

Confirm metrics are flowing to your custom namespace below.


In [ ]:
# List available metrics in our namespace
response = cloudwatch.list_metrics(Namespace=NAMESPACE)

print(f"Metrics in {NAMESPACE}:")
metric_names = set()
for metric in response.get('Metrics', []):
    metric_names.add(metric['MetricName'])

for name in sorted(metric_names):
    print(f"  - {name}")

print(f"\nTotal unique metrics: {len(metric_names)}")
print("\nYour dashboards:")
print(f"  Custom:     https://console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name={DASHBOARD_NAME}")
print(f"  GenAI Obs:  https://console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core/agents")


## Summary: What You've Built

You've instrumented your Lab 1 multi-agent workflow with production-ready monitoring. Your dashboard now tracks:

| Metric | Purpose | Action Threshold |
|--------|---------|------------------|
| **InputTokens** | Cost tracking | Monitor for unexpected spikes |
| **OutputTokens** | Cost tracking | 5× input rate — watch verbose agents |
| **CacheReadTokens** | Cost optimization | Higher = better (90% savings) |
| **LatencyMs** | User experience | >5000ms needs investigation |
| **CycleCount** | Prompt efficiency | >3 cycles = review agent prompts |
| **EstimatedCost** | Budget management | Set alerts for thresholds |
| **InvocationThrottles** | Capacity | >0 = request quota increase |

**Connecting to Lab 1:**
- Your `document_analysis` agent likely has high input tokens (processing claim descriptions)
- Your `claim_summary` agent likely has high output tokens (generating detailed summaries)
- The `inspection` agent may show variable latency depending on fraud pattern complexity

**Claude Haiku 4.5 Caching Notes:**
- Minimum 2,048 tokens per cache checkpoint
- 5-minute TTL (resets on each cache hit)
- Cache reads: 90% cost reduction
- Best for: system prompts in your agents, tool definitions, static context

> **💡 Why is the Cache Efficiency widget empty?** Prompt caching is not enabled in the workshop agents — the system prompts are short (~200 tokens, well below the 2,048-token checkpoint minimum) and each agent only runs a few times, so caching would have negligible impact. The widget is included so you know where to look when you enable caching in production. To enable it in Strands, pass `additional_request_fields={"anthropic_beta": ["prompt-caching-2024-07-31"]}` to `BedrockModel`.

**Next Steps:**
1. Set up CloudWatch Alarms for cost and latency thresholds
2. Enable prompt caching in your Lab 1 agents for cost optimization
3. Continue to **Lab 3** to measure agent accuracy with ground truth testing